# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ray007herowars-cmyk/Flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule

Rank content items using a simple search-opportunity score. The score gives more weight to search visibility and position opportunity, with CTR used as a supporting signal.

### Reason codes

- `SEARCH_OPPORTUNITY`: the page has meaningful search visibility and a position where improved CTR or ranking could be worth review.
- `PERFORMANCE_REVIEW`: the page receives lower search visibility but still warrants review based on the available performance signals.

The rule is decision-support only. It does not claim that changing a page will cause future performance to improve.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# W04 setup: load the March 2026 development feature frame

import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import userdata

# Hugging Face connection
token = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

# March 2026 warehouse relation
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# Build the same feature frame used in W03
features = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,

        AVG(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS avg_search_position,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_sessions
                ELSE 0
            END
        ) AS march_sessions,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN ga4_engaged_sessions
                ELSE 0
            END
        ) AS march_engaged_sessions

    FROM {REL}

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("Features available:")
print(features.columns.tolist())

print("\nRows:", len(features))

print("\nPreview:")
display(features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# W04 — Build baseline score and ranked queue

baseline = features.copy()

# CTR
baseline["ctr"] = np.where(
    baseline["march_impressions"] > 0,
    baseline["march_clicks"] / baseline["march_impressions"],
    np.nan
)

# Rank signals
volume_rank = baseline["march_impressions"].rank(pct=True)
position_rank = 1 - baseline["avg_search_position"].rank(pct=True)
ctr_rank = 1 - baseline["ctr"].rank(pct=True)

# Simple baseline score
baseline["score"] = (
    0.5 * volume_rank +
    0.3 * position_rank +
    0.2 * ctr_rank
)

# Reason code
baseline["reason_code"] = np.where(
    (baseline["march_impressions"] >= baseline["march_impressions"].median()) &
    (baseline["avg_search_position"] <= 20),
    "SEARCH_OPPORTUNITY",
    "PERFORMANCE_REVIEW"
)

# Action
baseline["action"] = np.where(
    baseline["reason_code"] == "SEARCH_OPPORTUNITY",
    "REVIEW",
    "INVESTIGATE"
)

# Rank
baseline = baseline.sort_values(
    ["score", "content_hash_id"],
    ascending=[False, True]
).reset_index(drop=True)

baseline["rank"] = range(1, len(baseline) + 1)

# Create output
output = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
]

# Save CSV
output_path = Path("/content/Flyrank-ML/work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print("Queue created successfully.")
print("Rows:", len(output))
print("Saved to:", output_path)

display(output.head(10))

### Signal verdicts

**Search volume — CONFIRMED:** Higher-impression buckets show higher median clicks, supporting search volume as a useful prioritization signal.

**CTR vs search position — MIXED:** Median CTR is 0.0 across all position buckets in this development slice, so the data does not show a clear directional relationship between search position and CTR.

These are directional checks only and do not establish that changing a page will cause future performance improvement.

## 3. Top-20 review

For each of the top 20 ranked content items, review the recommended action, the reason it received a high score, and what evidence could make the recommendation wrong.

The review is based only on March 2026 development-window features. It does not use future performance or outcome labels.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# W04 — Top-20 review

top20 = baseline.head(20).copy()

top20["confidence_note"] = np.where(
    top20["score"] >= top20["score"].quantile(0.75),
    "Higher-priority score based on available signals",
    "Moderate-priority score based on available signals"
)

top20["what_would_make_it_wrong"] = np.where(
    top20["march_impressions"] < baseline["march_impressions"].median(),
    "Low search volume may make the recommendation less useful",
    "The March signals may not represent current content performance"
)

review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

## 4. Weak picks + leakage check

The weaker-ranked items have lower baseline scores and are therefore less likely to be prioritized by this simple rule.

The baseline uses only March 2026 development-window performance features. No future-window data, outcome labels, or product flags were used as inputs.

The rule is intended for decision support and should be evaluated against later observed outcomes in a sealed test period.

In [ ]:
# W04 — Build baseline score and ranked queue

baseline = features.copy()

# CTR
baseline["ctr"] = np.where(
    baseline["march_impressions"] > 0,
    baseline["march_clicks"] / baseline["march_impressions"],
    0
)

# Replace missing signal values with neutral values
baseline["avg_search_position"] = baseline["avg_search_position"].fillna(
    baseline["avg_search_position"].median()
)

# Percentile ranks
volume_rank = baseline["march_impressions"].rank(pct=True)

position_rank = 1 - baseline["avg_search_position"].rank(pct=True)

ctr_rank = 1 - baseline["ctr"].rank(pct=True)

# Baseline score
baseline["score"] = (
    0.5 * volume_rank +
    0.3 * position_rank +
    0.2 * ctr_rank
)

# Reason code
baseline["reason_code"] = np.where(
    (baseline["march_impressions"] >= baseline["march_impressions"].median()) &
    (baseline["avg_search_position"] <= 20),
    "SEARCH_OPPORTUNITY",
    "PERFORMANCE_REVIEW"
)

# Action
baseline["action"] = np.where(
    baseline["reason_code"] == "SEARCH_OPPORTUNITY",
    "REVIEW",
    "INVESTIGATE"
)

# Rank
baseline = baseline.sort_values(
    ["score", "content_hash_id"],
    ascending=[False, True]
).reset_index(drop=True)

baseline["rank"] = range(1, len(baseline) + 1)

# Output
output = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action"
    ]
]

# Save CSV
output_path = Path("/content/Flyrank-ML/work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print("Queue created successfully.")
print("Rows:", len(output))
print("NaN scores:", output["score"].isna().sum())
print("Saved to:", output_path)

display(output.head(10))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.